# auto_packet_analyze_v3 — Kaggle 파이프라인

pcap 하나(또는 여러 개)를 넣으면 **extract_log → evidence → LLM 분석**까지 자동으로 돌린다.

## 실행 전 준비 (필수)
1. 우측 **Settings → Internet: ON** (apt/ollama/모델 다운로드에 필요)
2. 우측 **Settings → Accelerator: GPU** (gemma4:26b 는 GPU 필요)
3. **Add Input → 업로드한 pcap 데이터셋 연결** (pcap 은 `/kaggle/input/<dataset>/` 에 놓임)

> 첫 실행은 오래 걸린다(suricata/zeek/ollama 설치 + 모델 ~16GB pull). 세션이 휘발성이라 매 세션 반복됨.


## 1. 설정 + 레포 clone

In [ ]:
import os, subprocess, glob, json

REPO_URL = "https://github.com/DAADAISMYLIFE/auto_packet_analyze_v3.git"
WORK = "/kaggle/working"
REPO = f"{WORK}/auto_packet_analyze_v3"
MODEL = "gemma4:26b"          # 필요하면 변경 (실존 모델로)
os.environ["MODEL"] = MODEL

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)
print("repo :", REPO)
print("model:", MODEL)


## 2. 환경 구성 (suricata/zeek/ollama 설치 + 모델 pull)
`setup.sh` 가 전부 처리한다. **오래 걸림.**

In [ ]:
# Kaggle 은 root 라 sudo 없이 설치됨. 모델 pull + test.py 스모크까지 수행.
# {REPO}/{MODEL} 는 Jupyter 가 위 셀의 파이썬 변수로 치환한다.
!cd {REPO} && MODEL={MODEL} bash setup.sh


## 3. ollama 서버 + 파이썬 패키지 확인

In [ ]:
# setup.sh 가 서버를 띄우지만, 세션 상황에 따라 재확인
import subprocess, time, urllib.request

def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=3)
        return True
    except Exception:
        return False

if not ollama_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                     stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)

print("ollama up:", ollama_up())
!ollama list
!pip install -q ollama   # run.py 가 쓰는 파이썬 패키지


## 4. 파이프라인 실행 (업로드한 pcap 자동 탐색)
`/kaggle/input/**/*.pcap` 를 찾아 각각 extract → evidence → run (LLM 분석).

In [ ]:
import glob, os, subprocess

pcaps = sorted(glob.glob("/kaggle/input/**/*.pcap", recursive=True))
print("발견된 pcap:", pcaps or "(없음 — Add Input 으로 데이터셋 연결했는지 확인)")

env = dict(os.environ, MODEL=MODEL)
for p in pcaps:
    name = os.path.splitext(os.path.basename(p))[0]
    print("\n" + "=" * 60 + f"\n  {name}\n" + "=" * 60)

    # 3-1) suricata + zeek 로그 추출
    subprocess.run(["bash", "scripts/extract_log.sh", p], cwd=REPO, check=False)
    # 3-2) evidence.json (정규화/압축)
    subprocess.run(["python", "scripts/build_evidence.py", name], cwd=REPO, check=False)
    # 3-3) LLM 분석 (run.py — 결과는 이 셀 stdout 으로 출력)
    subprocess.run(["python", "llm/run.py", name], cwd=REPO, env=env, check=False)


## 5. 결과 확인 (evidence + analysis)

In [ ]:
import glob, json

for name_dir in sorted(glob.glob(f"{REPO}/output/*/")):
    name = os.path.basename(name_dir.rstrip("/"))
    ev = os.path.join(name_dir, "evidence.json")
    an = os.path.join(name_dir, "analysis.json")
    print("=" * 70, f"\n{name}")
    if os.path.isfile(ev):
        e = json.load(open(ev))
        print("[evidence] hosts=%d alerts=%d files=%d ext_dom=%d"
              % (len(e["hosts"]), len(e["alerts"]), len(e["files"]),
                 len(e["external"]["domains"])))
    if os.path.isfile(an):
        print("[analysis.json]")
        print(json.dumps(json.load(open(an)), ensure_ascii=False, indent=2))
    else:
        print("(run.py 는 분석 결과를 위 파이프라인 셀 stdout 으로 출력함 — 거기서 확인)")
